In [ ]:
# Resolve repository imports and data paths when opened from notebooks/.
from pathlib import Path
import os
import sys

repo_root = Path.cwd().resolve()
while not (repo_root / "sae").is_dir():
    if repo_root == repo_root.parent:
        raise FileNotFoundError("Run this notebook from the KronSAE repository.")
    repo_root = repo_root.parent

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from tqdm.notebook import tqdm

import torch
import numpy as np
    
from transformer_lens import HookedTransformer
import transformer_lens.utils as utils
from transformers import AutoConfig
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from safetensors.torch import load_model
##########
import torch
import json
import math
import einops 
import numpy as np
import torch.nn.functional as F

from torch import nn
from transformer_lens import HookedTransformer
from transformer_lens import utils

from sae import KronSAE, TopKSAE
from sae.config import TrainingConfig, SAEConfig

from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

import transformer_lens
import transformer_lens.utils as utils
from transformer_lens.hook_points import (
    HookPoint,
)  # Hooking utilities
from transformer_lens import HookedTransformer, FactoredMatrix

import einops
from fancy_einsum import einsum
import tqdm.auto as tqdm
import plotly.express as px

from functools import partial

from huggingface_hub.hf_api import HfFolder

device = "cuda"

def get_sae(sae_type: str, path):
    # Load transformer model
    sae_class = {
        "topk": TopKSAE,
        "kronsae": KronSAE
    }[sae_type]
    
    sae_config = SAEConfig.from_pretrained(path)
    sae_config.dtype = "float32"
    sae_config.sae_dtype = "float32"

    sae = sae_class(sae_config)
    load_model(sae, f"{path}/model.safetensors")
    sae.to(device)
    return {'sae': sae, 'config': sae_config}
    
sae_paths = {
    "topk" : "saes/16k_topk", 
    "kronsae" : "saes/16k_kronsae_4_16",
}
d = get_sae("topk", "saes/1_5b_qwen/32k_topk")
topk_sae = d["sae"]
topk_sae_config = d["config"]


In [ ]:
import torch
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

A = torch.load('mul_corr.pt', map_location='cpu').float().numpy()
B = torch.load('topk_corr.pt', map_location='cpu').float().numpy()

np.fill_diagonal(A, 0)
np.fill_diagonal(B, 0)

hist_mul = np.sort(A, axis=1).mean(axis=0)
hist_topk = np.sort(B, axis=1).mean(axis=0)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming hist_mul and hist_topk are your data arrays
fig, ax = plt.subplots(1, 1, figsize=(8, 3), dpi=250)

palette = sns.color_palette('Set2', 2)

# Plot both positive and negative values together using symlog scale
sns.histplot(hist_mul,   log_scale=False, stat="density", bins=256 + 128, 
             label="KronSAE", ax=ax, element="step", fill=False, color=palette[0])
sns.histplot(hist_topk, log_scale=False, stat="density", bins=256 + 128, 
             label="TopK", ax=ax, element="step", fill=False, color=palette[1])

# Set symmetric logarithmic x-axis
ax.set_xscale('symlog', linthresh=2e-3)  # Adjust linthresh based on your data

# scales = [1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8, 1e-9, 1e-10, 1e-11]
# scales = [-s for s in scales] + [0] + list(reversed(scales))

# Customize ticks to show negative/positive explicitly
# ax.set_xticks(scales)
# ax.set_xticklabels([f'{s:.0e}' for s in scales])

# Log scale for y-axis to see density differences
ax.set_yscale('log')
ax.grid(alpha=0.35)
# ax.set_title('Positive & Negative Values (Symmetric Log Scale)')
ax.legend(loc='upper right')

# plt.axvline(np.median(hist_mul), linestyle='--',  color=palette[0])
# plt.axvline(np.median(hist_topk), linestyle='--',  color=palette[1])

plt.tight_layout()
plt.show()

In [ ]:
print(f"KronSAE: mean {hist_mul.mean():.4e}, median {np.median(hist_mul):.5f}")
print(f"TopK: mean {hist_topk.mean():.4e}, median {np.median(hist_topk):.5f}")

In [ ]:
import torch
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

# Example usage:
corr_matrix = torch.tensor(A)
group_size = 16
max_samples: int = 100000
random_seed: int = 42
"""
Compare within-group and between-group correlation distributions.

Parameters:
- corr_matrix: torch.Tensor of shape (N, N), correlation matrix.
- group_size: int, size k of each group (assumed contiguous blocks).
- max_samples: int, maximum number of samples per distribution to plot/compare.
- random_seed: int, for reproducibility of sampling.

Returns:
- t_stat: float, t-test statistic comparing inner vs. between.
- p_value: float, two-sided p-value of the test.
"""
torch.manual_seed(random_seed)
N = corr_matrix.size(0)
if N % group_size != 0:
    raise ValueError("Matrix size must be divisible by group_size")
num_groups = N // group_size

inner_vals = []
between_vals = []

# Collect samples
for g in range(num_groups):
    s1 = g * group_size
    e1 = s1 + group_size
    block = corr_matrix[s1:e1, s1:e1]
    triu = torch.triu_indices(group_size, group_size, offset=1)
    inner_vals.append(block[triu[0], triu[1]])

    # sample sqrt(num_groups) other groups
    others = list(range(num_groups))
    others.remove(g)
    k_sample = min(len(others), int(np.sqrt(num_groups)))
    chosen = np.random.choice(others, size=k_sample, replace=False)
    for h in chosen:
        s2 = h * group_size
        e2 = s2 + group_size
        between_vals.append(corr_matrix[s1:e1, s2:e2].flatten())

inner = torch.cat(inner_vals)
between = torch.cat(between_vals)

# subsample if needed
def subsample(tensor):
    total = tensor.numel()
    if total > max_samples:
        idx = torch.randperm(total)[:max_samples]
        return tensor[idx]
    return tensor

inner = subsample(inner)
between = subsample(between)

inner_np = inner.cpu().numpy()
between_np = between.cpu().numpy()

combined = np.concatenate((inner_np, between_np))
min_val, max_val = combined.min(), combined.max()
num_bins = 36
bins = np.linspace(min_val, max_val, num_bins + 1)


In [ ]:


# # Plot both histograms on one axis with same bins
# plt.figure(figsize=(8, 6))
# sns.histplot(inner_np, bins=bins, kde=False, stat="density",
#              label='Within-Group', alpha=0.6)
# sns.histplot(between_np, bins=bins, kde=False, stat="density",
#              label='Between-Group', alpha=0.6)

In [ ]:
# Plot both histograms on one axes
plt.figure(figsize=(8, 6))
sns.histplot(inner_np, bins=bins,  kde=False, stat="density", label='Within-Group', alpha=0.6)
sns.histplot(between_np, bins=bins,  kde=False, stat="density", label='Between-Group', alpha=0.6)

# Avoid zero-density dominating by setting bins away from zero and using logarithmic scales
#plt.xscale('symlog')
plt.yscale('log')
plt.xlabel('Correlation Value')
plt.ylabel('Density (log scale)')
plt.title('Within vs. Between Group Correlation Distributions')
plt.legend()
plt.tight_layout()
plt.show()

# t-test
t_stat, p_value = ttest_ind(inner_np, between_np, equal_var=False)
print(f"T-statistic: {t_stat:.4f}, p-value: {p_value:.4e}")


In [ ]:
print(f"KronSAE: inner corr mean {inner_np.mean():.4e}, median {np.median(inner_np):.5f}")
print(f"KronSAE: inner between mean {between_np.mean():.4e}, median {np.median(between_np):.5f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming hist_mul and hist_topk are your data arrays
fig, ax = plt.subplots(1, 2, figsize=(10, 3), dpi=250)

palette = sns.color_palette('Set2', 2)

# Plot both positive and negative values together using symlog scale
sns.histplot(hist_mul,   log_scale=False, stat="density", bins=256 + 128, 
             label="KronSAE", ax=ax[0], element="step", fill=False, color=palette[0])
sns.histplot(hist_topk, log_scale=False, stat="density", bins=256 + 128, 
             label="TopK", ax=ax[0], element="step", fill=False, color=palette[1])

# Set symmetric logarithmic x-axis
ax[0].set_xscale('symlog', linthresh=2e-3)  # Adjust linthresh based on your data
ax[0].set_ylabel('Density (log scale)')

# scales = [1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8, 1e-9, 1e-10, 1e-11]
# scales = [-s for s in scales] + [0] + list(reversed(scales))

# Customize ticks to show negative/positive explicitly
# ax.set_xticks(scales)
# ax.set_xticklabels([f'{s:.0e}' for s in scales])

# Log scale for y-axis to see density differences
ax[0].set_yscale('log')
ax[0].grid(alpha=0.35)
# ax.set_title('Positive & Negative Values (Symmetric Log Scale)')
ax[0].legend(loc='upper right')

# plt.axvline(np.median(hist_mul), linestyle='--',  color=palette[0])
# plt.axvline(np.median(hist_topk), linestyle='--',  color=palette[1])

sns.histplot(inner_np, bins=bins, ax=ax[1], kde=False, stat="density", label='Within-Group', alpha=0.6)
sns.histplot(between_np, bins=bins, ax=ax[1], kde=False, stat="density", label='Between-Group', alpha=0.6)

# Avoid zero-density dominating by setting bins away from zero and using logarithmic scales
#plt.xscale('symlog')
ax[1].set_yscale('log')
#ax[1].set_xlabel('Correlation Value')
ax[1].set_ylabel('')
#ax[1].set_title('Within vs. Between Group Correlation Distributions')
ax[1].grid(alpha=0.35)
fig.text(0.5, -0.00, 'Correlation Value', ha='center')
plt.legend()
plt.tight_layout()
plt.savefig(f'corrs.pdf')
plt.show()
